# 02 Generate FACT Data
This notebook generates transactional data (Invoices) linked to the DIM snapshots, strictly following KiotViet templates.

In [ ]:
import pandas as pd
import numpy as np
import os
import json
from datetime import datetime, timedelta
from faker import Faker

fake = Faker('vi_VN')

# Scenario Configuration
CONFIG = {
    'date_range': ('2026-03-01', '2026-03-31'),
    'invoices_per_day': (10, 30),
    'basket_size': (1, 6),
    'seed': 42
}

np.random.seed(CONFIG['seed'])

HEADERS = [
    'Mã hóa đơn', 'Thời gian', 'Người bán', 'Kênh bán', 'Mã khách hàng', 
    'Tên khách hàng', 'Điện thoại', 'Email', 'Địa chỉ (Khách hàng)', 
    'Khu vực (Khách hàng)', 'Phường/Xã (Khách hàng)', 'Bảng giá', 
    'Mã hàng', 'Số lượng', 'Đơn giá', 'Giảm giá %', 'Giảm giá', 
    'Giảm giá hóa đơn', 'Giảm giá hóa đơn %', 'Tiền mặt', 'Thẻ', 
    'Chuyển khoản', 'Ghi chú'
]

print("FACT Generation Configured with exact KiotViet headers.")

## 1. Load DIM Snapshots

In [ ]:
df_products = pd.read_csv('../data/out/csv/dim_products.csv')
df_customers = pd.read_csv('../data/out/csv/dim_customers.csv')
df_employees = pd.read_csv('../data/out/csv/dim_employees.csv')

print(f"Syncing with DIMs: {len(df_products)} products, {len(df_customers)} customers.")

## 2. Generate Flat-File Invoices
Generating rows where header info is duplicated across product lines (KiotViet standard).

In [ ]:
all_rows = []
start_date = datetime.strptime(CONFIG['date_range'][0], '%Y-%m-%d')
end_date = datetime.strptime(CONFIG['date_range'][1], '%Y-%m-%d')
delta = end_date - start_date

invoice_id_counter = 1

for i in range(delta.days + 1):
    current_date = start_date + timedelta(days=i)
    daily_count = np.random.randint(*CONFIG['invoices_per_day'])
    
    for _ in range(daily_count):
        invoice_code = f"HDIP{invoice_id_counter:05d}"
        customer = df_customers.sample(1).iloc[0]
        seller = df_employees.sample(1).iloc[0]
        time_str = (current_date + timedelta(hours=np.random.randint(8, 21), 
                                           minutes=np.random.randint(0, 59))).strftime('%Y-%m-%d %H:%M:%S')
        
        num_lines = np.random.randint(*CONFIG['basket_size'])
        
        # Multi-table relationship: Picking products and quantities
        invoice_total = 0
        lines = []
        for _ in range(num_lines):
            prod = df_products.sample(1).iloc[0]
            qty = np.random.randint(1, 4)
            price = prod['Giá bán']
            invoice_total += (price * qty)
            lines.append((prod['Mã hàng'], qty, price))

        # Fill rows for KiotViet import
        for j, (p_code, p_qty, p_price) in enumerate(lines):
            row = {h: np.nan for h in HEADERS}
            row['Mã hóa đơn'] = invoice_code
            row['Thời gian'] = time_str
            row['Người bán'] = seller['Tên nhân viên (*)']
            row['Kênh bán'] = 'Bán trực tiếp'
            row['Mã khách hàng'] = customer['Mã khách hàng']
            row['Tên khách hàng'] = customer['Tên khách hàng']
            row['Điện thoại'] = customer['Điện thoại']
            row['Mã hàng'] = p_code
            row['Số lượng'] = p_qty
            row['Đơn giá'] = p_price
            
            # Only first line shows payment (KiotViet rule often requires this or specific allocation)
            if j == 0:
                row['Tiền mặt'] = invoice_total
            
            all_rows.append(row)
        
        invoice_id_counter += 1

df_final = pd.DataFrame(all_rows)
print(f"Synthesized {len(df_final)} rows for {invoice_id_counter-1} invoices.")

## 3. Export to Files

In [ ]:
output_excel = '../data/out/excel/FACT_INVOICES_Full.xlsx'
df_final.to_excel(output_excel, index=False, sheet_name='InvoiceTemplate')

df_final.to_csv('../data/out/csv/fact_invoices_flat.csv', index=False)

print(f"FACT data ready at: {output_excel}")